## RGDI 可解释性分析

从训练好的 ADSSTNet checkpoint 前向推理，收集各 encoder 层 **RGDI** 的中间特征：

| 键名 | 含义 | 形状 |
|------|------|------|
| `mambaX` | DI Block 前，时序 Mamba 分支输出 | `(L, D)` |
| `gcnX` | DI Block 前，空间 GCN 分支输出 | `(L, D)` |
| `mambaX_` | DI Block 后，差分抑制后的 Mamba 特征 | `(L, D)` |
| `gcnX_` | DI Block 后，差分抑制后的 GCN 特征 | `(L, D)` |

其中 `L` 为当前层序列长度（多尺度下逐层折半），`D = d_model`。

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import torch

import random
from argparse import Namespace

from torch.optim.swa_utils import AveragedModel
from torch.utils.data import DataLoader

from data_provider.data_factory import data_provider
from data_provider.uea import collate_fn
from models import ADSSTNet
from utils.tools import compute_pearson_A0_subject_mean

# 将项目根目录加入 sys.path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'run.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========== 路径与输出配置 ==========
save_path = PROJECT_ROOT / 'figs' / 'Weight Visualization Analysis'
DI_OUTPUT_DIR = save_path / 'Residual-Guided Differential Interaction'
DI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 20 通道标准 10-20 名称（与 CAEEG 通道顺序一致）
CH_NAMES = [
    'Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'Cz', 'C3', 'C4',
    'T7', 'T8', 'Pz', 'P3', 'P4', 'P7', 'P8', 'Oz', 'O1', 'O2',
]

# 与训练脚本 ADSSTNet-Ablation-Experiment.sh 一致
INPUT_SEQ_LEN = 128
DOWNSAMPLE_MIN_LEN = 4

print(f'PROJECT_ROOT   = {PROJECT_ROOT}')
print(f'DI_OUTPUT_DIR  = {DI_OUTPUT_DIR}')

PROJECT_ROOT   = /root/autodl-tmp/ADSST-Net
DI_OUTPUT_DIR  = /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction


In [2]:
def _as_layer_list(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]


def compute_seq_each_layer(seq_len: int, downsample_min_len: int | None) -> list[int]:
    """复现 Encoder 中 seq_each_layer 的计算逻辑（层间 AvgPool(2) 折半）。"""
    if downsample_min_len is None or downsample_min_len >= seq_len:
        return [seq_len]
    seq_list = [seq_len]
    L = seq_len
    while L > downsample_min_len:
        L = max(1, (L - 2) // 2 + 1)
        seq_list.append(L)
    return seq_list


def _get_data(flag: str, fold_seed: int):
    fold_args = Namespace(**MODEL_ARGS_BASE, seed=fold_seed)
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    return data_provider(fold_args, flag, log=None)


def build_adsstnet_for_fold(fold_seed: int):
    fold_args = Namespace(**MODEL_ARGS_BASE, seed=fold_seed)
    random.seed(fold_seed)
    np.random.seed(fold_seed)

    train_data, _ = data_provider(fold_args, 'TRAIN', log=None)
    fold_args.seq_len = train_data.max_seq_len
    fold_args.enc_in = train_data.X.shape[-1]
    if fold_args.selected_classes:
        fold_args.num_class = len(fold_args.selected_classes)
    else:
        fold_args.num_class = len(np.unique(train_data.y[:, 0]))
    fold_args.A0 = compute_pearson_A0_subject_mean(
        train_data, enc_in=fold_args.enc_in, **A0_KWARGS,
    )

    model = ADSSTNet.Model(fold_args).float().to(DEVICE)
    if USE_SWA:
        model = AveragedModel(model)
    model.eval()
    return model, fold_args


def _make_loader(fold_args, flag: str):
    dataset, _ = _get_data(flag, fold_args.seed)
    loader = DataLoader(
        dataset,
        batch_size=fold_args.batch_size,
        shuffle=False,
        num_workers=fold_args.num_workers,
        drop_last=False,
        collate_fn=lambda x: collate_fn(x, max_len=fold_args.seq_len),
    )
    return dataset, loader


def load_train_loader(fold_args):
    return _make_loader(fold_args, 'TRAIN')


def load_test_loader(fold_args):
    return _make_loader(fold_args, 'TEST')


def load_fold_checkpoint(model, fold_dir: Path):
    state = torch.load(fold_dir / 'checkpoint.pth', map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()


def _balance_class_indices(labels, class_ids, rng: np.random.Generator):
    labels = np.asarray(labels)
    class_ids = sorted(class_ids)
    per_class_idx = {c: np.where(labels == c)[0] for c in class_ids}
    counts = {c: len(per_class_idx[c]) for c in class_ids}
    missing = [c for c in class_ids if counts[c] == 0]
    if missing:
        raise RuntimeError(f'类别 {missing} 无样本，无法均衡采样')

    min_n = min(counts.values())
    selected = []
    for c in class_ids:
        idx = per_class_idx[c]
        chosen = idx if len(idx) == min_n else rng.choice(idx, size=min_n, replace=False)
        selected.extend(int(i) for i in chosen)
    rng.shuffle(selected)
    return selected, min_n, counts


def _subset_diff_samples(data_by_layer, label_list, keep_indices):
    label_list = [label_list[i] for i in keep_indices]
    subset = {
        layer_idx: {
            key: [data_by_layer[layer_idx][key][i] for i in keep_indices]
            for key in DIFF_KEYS
        }
        for layer_idx in data_by_layer
    }
    return subset, label_list


@torch.no_grad()
def collect_differential_visualization(
    model,
    data_loader,
    class_names,
    correct_only: bool = True,
    balance_classes: bool = True,
    balance_seed=None,
    class_ids=None,
    split_name: str = 'TEST',
):
    """在指定 DataLoader 上收集全部 encoder 层的 RGDI_visualization。

    Returns
    -------
    data_by_layer : dict[int, dict[str, list[np.ndarray]]]
        每层包含 DIFF_KEYS 四个键，值为样本列表，单样本 shape=(L, D)。
    label_list : list[int]
    layer_ids : list[int]
    """
    data_by_layer = None
    label_list = []
    n_total = n_kept = 0

    for batch_x, label_id, _ in data_loader:
        batch_x_ori = batch_x[:, 0, :, :].float().to(DEVICE)
        labels = label_id[:, 0].cpu().numpy().astype(int)
        n_samples = len(labels)
        n_total += n_samples

        _, logits, _, _, _, rgdi_vis_list = model(
            batch_x_ori, None, None, None, model_state='test',
        )
        preds = logits.argmax(dim=-1).detach().cpu().numpy().astype(int)
        sample_indices = (
            [i for i in range(n_samples) if preds[i] == labels[i]]
            if correct_only else list(range(n_samples))
        )
        if not sample_indices:
            continue
        n_kept += len(sample_indices)

        rgdi_layers = _as_layer_list(rgdi_vis_list)
        n_layers = len(rgdi_layers)

        if data_by_layer is None:
            data_by_layer = {
                layer_idx: {k: [] for k in DIFF_KEYS}
                for layer_idx in range(n_layers)
            }

        for layer_idx in range(n_layers):
            rgdi_dict = rgdi_layers[layer_idx]
            for key in DIFF_KEYS:
                if key not in rgdi_dict:
                    raise KeyError(
                        f'Layer {layer_idx} RGDI_visualization 缺少键 {key!r}，'
                        f'现有键: {list(rgdi_dict.keys())}'
                    )
                tensor = rgdi_dict[key]
                for i in sample_indices:
                    data_by_layer[layer_idx][key].append(
                        tensor[i].detach().cpu().numpy()
                    )
        label_list.extend(int(labels[i]) for i in sample_indices)

    if data_by_layer is None:
        raise RuntimeError(f'[{split_name}] 未收集到任何样本')

    prefix = f'collect_differential_visualization [{split_name}]'
    if correct_only:
        print(f'{prefix}: 保留正确分类样本 {n_kept}/{n_total}')
    else:
        print(f'{prefix}: 保留全部样本 {n_kept}/{n_total}')

    if balance_classes:
        balance_class_ids = sorted(set(class_ids if class_ids is not None else label_list))
        rng = np.random.default_rng(balance_seed)
        keep_indices, min_n, raw_counts = _balance_class_indices(
            label_list, balance_class_ids, rng,
        )
        orig_labels = label_list
        data_by_layer, label_list = _subset_diff_samples(
            data_by_layer, orig_labels, keep_indices,
        )
        count_str = ', '.join(
            f'{class_names[c] if c < len(class_names) else c}: {raw_counts[c]}->{min_n}'
            for c in balance_class_ids
        )
        print(f'{prefix}: 类别均衡采样 -> 每类 n={min_n} ({count_str})')

    layer_ids = sorted(data_by_layer.keys())
    return data_by_layer, label_list, layer_ids


def summarize_differential_data(data_by_layer, label_list, layer_ids, class_names, split_name: str = 'TEST'):
    """打印各层各键的样本数与单样本 shape。"""
    print(f'\n=== {split_name} 数据摘要 ===')
    print(f'总样本数: {len(label_list)}')
    if label_list:
        labels = np.asarray(label_list)
        for c in sorted(set(labels)):
            name = class_names[c] if c < len(class_names) else str(c)
            print(f'  {name}: {int((labels == c).sum())}')
    for layer_idx in layer_ids:
        sample0 = data_by_layer[layer_idx][DIFF_KEYS[0]][0]
        n = len(data_by_layer[layer_idx][DIFF_KEYS[0]])
        print(f'Layer {layer_idx}: n={n}, shape={sample0.shape} (L, D)')
        for key in DIFF_KEYS:
            arr = np.stack(data_by_layer[layer_idx][key], axis=0)
            print(
                f'  {key:8s} mean={arr.mean():.4f}, std={arr.std():.4f}, '
                f'min={arr.min():.4f}, max={arr.max():.4f}'
            )


import pandas as pd
from sklearn.utils import resample
from sklearn.metrics import silhouette_score

DEFAULT_CLASS_PAIRS_2 = ((0, 1),)
DEFAULT_CLASS_PAIRS_3 = ((0, 1), (0, 2), (1, 2))
DI_BRANCH_METRIC_CONFIGS = (
    dict(branch='mamba', before_key='mambaX', after_key='mambaX_', title='Mamba branch'),
    dict(branch='gcn', before_key='gcnX', after_key='gcnX_', title='GCN branch'),
)


def hedges_g(x1, x2):
    """Calculate Hedges' g effect size between two groups."""
    x1 = np.asarray(x1, dtype=np.float64)
    x2 = np.asarray(x2, dtype=np.float64)
    n1 = len(x1)
    n2 = len(x2)
    if n1 < 2 or n2 < 2:
        return np.nan
    mean1 = np.mean(x1)
    mean2 = np.mean(x2)
    std1 = np.std(x1, ddof=1)
    std2 = np.std(x2, ddof=1)
    pooled_std = np.sqrt(
        ((n1 - 1) * std1**2 + (n2 - 1) * std2**2)
        / (n1 + n2 - 2)
    )
    if pooled_std == 0:
        return np.nan
    d = (mean1 - mean2) / pooled_std
    correction = 1 - (3 / (4 * (n1 + n2) - 9))
    return correction * d


def bootstrap_hedges_g(x1, x2, n_boot=5000):
    values = []
    x1 = np.asarray(x1, dtype=np.float64)
    x2 = np.asarray(x2, dtype=np.float64)
    for _ in range(n_boot):
        b1 = resample(x1)
        b2 = resample(x2)
        g = hedges_g(b1, b2)
        if not np.isnan(g):
            values.append(g)
    if not values:
        return np.nan, (np.nan, np.nan)
    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)
    return np.mean(values), (lower, upper)


def fisher_score(X, y):
    """Multi-class Fisher Score; X shape (N, D), y shape (N,)."""
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)
    classes = np.unique(y)
    overall_mean = np.mean(X, axis=0)
    numerator = np.zeros(X.shape[1], dtype=np.float64)
    denominator = np.zeros(X.shape[1], dtype=np.float64)
    for c in classes:
        Xc = X[y == c]
        nc = len(Xc)
        mean_c = np.mean(Xc, axis=0)
        var_c = np.var(Xc, axis=0)
        numerator += nc * (mean_c - overall_mean) ** 2
        denominator += nc * var_c
    fisher_per_feature = numerator / (denominator + 1e-12)
    return float(np.mean(fisher_per_feature))


def compute_silhouette_score(features, labels, metric='euclidean'):
    features = np.asarray(features, dtype=np.float64)
    labels = np.asarray(labels)
    n_classes = len(np.unique(labels))
    if len(labels) < 2 or n_classes < 2 or len(labels) <= n_classes:
        return np.nan
    return float(silhouette_score(features, labels, metric=metric))


def _pair_label(class_names, c1, c2):
    name1 = class_names[c1] if c1 < len(class_names) else str(c1)
    name2 = class_names[c2] if c2 < len(class_names) else str(c2)
    return f'{name1} vs {name2}'


def mean_abs_hedges_g_per_pair(features, labels, c1, c2):
    """各特征维度的 |Hedges' g| 均值，作为多变量特征的效应量摘要。"""
    labels = np.asarray(labels)
    x1 = features[labels == c1]
    x2 = features[labels == c2]
    if len(x1) < 2 or len(x2) < 2:
        return np.nan
    gs = []
    for d in range(features.shape[1]):
        g = hedges_g(x1[:, d], x2[:, d])
        if not np.isnan(g):
            gs.append(abs(g))
    return float(np.mean(gs)) if gs else np.nan


def bootstrap_mean_abs_hedges_g(features, labels, c1, c2, n_boot=5000):
    labels = np.asarray(labels)
    x1 = features[labels == c1]
    x2 = features[labels == c2]
    if len(x1) < 2 or len(x2) < 2:
        return np.nan, (np.nan, np.nan)
    values = []
    for _ in range(n_boot):
        b1 = resample(x1)
        b2 = resample(x2)
        gs = []
        for d in range(features.shape[1]):
            g = hedges_g(b1[:, d], b2[:, d])
            if not np.isnan(g):
                gs.append(abs(g))
        if gs:
            values.append(np.mean(gs))
    if not values:
        return np.nan, (np.nan, np.nan)
    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)
    return float(np.mean(values)), (float(lower), float(upper))


def _subset_pair_features(features, labels, c1, c2):
    labels = np.asarray(labels, dtype=int)
    mask = (labels == c1) | (labels == c2)
    return features[mask], labels[mask]


def compute_pairwise_metrics(features, labels, class_pairs, class_names, n_boot=5000):
    """两两类别对比：Fisher Score、Silhouette Score、Effect Size 对齐计算。"""
    features = np.asarray(features, dtype=np.float64)
    labels = np.asarray(labels, dtype=int)
    records = []
    for c1, c2 in class_pairs:
        X_pair, y_pair = _subset_pair_features(features, labels, c1, c2)
        g = mean_abs_hedges_g_per_pair(features, labels, c1, c2)
        g_boot, (lo, hi) = bootstrap_mean_abs_hedges_g(features, labels, c1, c2, n_boot=n_boot)
        fisher_val = fisher_score(X_pair, y_pair) if len(X_pair) >= 4 else np.nan
        records.append({
            'pair': _pair_label(class_names, c1, c2),
            'c1': int(c1),
            'c2': int(c2),
            'fisher_score': fisher_val,
            'silhouette_score': compute_silhouette_score(X_pair, y_pair),
            'mean_abs_hedges_g': g,
            'mean_abs_hedges_g_boot': g_boot,
            'ci_low': lo,
            'ci_high': hi,
        })
    return records


def compute_feature_metrics(features, labels, class_names, class_pairs=None, n_boot=5000):
    """单组特征：两两类别对比的 Fisher / Silhouette / Effect Size。"""
    features = np.asarray(features, dtype=np.float64)
    labels = np.asarray(labels, dtype=int)
    if class_pairs is None:
        class_pairs = DEFAULT_CLASS_PAIRS_3
    return {
        'pairwise_metrics': compute_pairwise_metrics(
            features, labels, class_pairs, class_names, n_boot=n_boot,
        ),
    }


def compute_di_stage_metrics(diff_by_layer, labels, layer_idx, key, class_names, n_boot=5000, class_pairs=None):
    """单层单阶段（DI Block 前或后）特征指标。"""
    features = diff_key_to_features(diff_by_layer, layer_idx, key)
    return compute_feature_metrics(
        features, labels, class_names, class_pairs=class_pairs, n_boot=n_boot,
    )


def compute_di_block_metrics_table(
    diff_by_layer,
    labels,
    layer_ids,
    class_names,
    before_key,
    after_key,
    branch_name,
    class_pairs=None,
    n_boot=5000,
):
    rows = []
    labels = np.asarray(labels, dtype=int)
    for layer_idx in layer_ids:
        before = compute_di_stage_metrics(
            diff_by_layer, labels, layer_idx, before_key, class_names,
            n_boot=n_boot, class_pairs=class_pairs,
        )
        after = compute_di_stage_metrics(
            diff_by_layer, labels, layer_idx, after_key, class_names,
            n_boot=n_boot, class_pairs=class_pairs,
        )
        row = {
            'branch': branch_name,
            'layer': layer_idx,
            'before_key': before_key,
            'after_key': after_key,
            'pairwise_before': before['pairwise_metrics'],
            'pairwise_after': after['pairwise_metrics'],
        }
        rows.append(row)
    return rows


def _format_pairwise_metric_line(rec):
    return (
        f"      {rec['pair']:12s}: "
        f"Fisher={rec['fisher_score']:.4f}, "
        f"Silhouette={rec['silhouette_score']:.4f}, "
        f"mean|g|={rec['mean_abs_hedges_g']:.4f}, "
        f"boot={rec['mean_abs_hedges_g_boot']:.4f}, "
        f"95%CI=[{rec['ci_low']:.4f}, {rec['ci_high']:.4f}]"
    )


def _di_metrics_to_dataframe(all_results, fold_tag, split_name):
    """将指标结果转为单表 DataFrame（两两类别对比，DI 前后对齐）。"""
    rows = []
    for branch, table_rows in all_results.items():
        for row in table_rows:
            for stage, pairwise_metrics, feature_key in (
                ('before', row['pairwise_before'], row['before_key']),
                ('after', row['pairwise_after'], row['after_key']),
            ):
                for rec in pairwise_metrics:
                    rows.append({
                        'split': split_name,
                        'fold': fold_tag,
                        'branch': branch,
                        'layer': row['layer'],
                        'stage': stage,
                        'feature_key': feature_key,
                        'pair': rec['pair'],
                        'c1': rec['c1'],
                        'c2': rec['c2'],
                        'fisher_score': rec['fisher_score'],
                        'silhouette_score': rec['silhouette_score'],
                        'mean_abs_hedges_g': rec['mean_abs_hedges_g'],
                        'mean_abs_hedges_g_boot': rec['mean_abs_hedges_g_boot'],
                        'ci_low': rec['ci_low'],
                        'ci_high': rec['ci_high'],
                    })

    df_metrics = pd.DataFrame(rows)
    float_cols = [
        'fisher_score', 'silhouette_score',
        'mean_abs_hedges_g', 'mean_abs_hedges_g_boot', 'ci_low', 'ci_high',
    ]
    if not df_metrics.empty:
        df_metrics[float_cols] = df_metrics[float_cols].round(6)
    return df_metrics


def save_di_block_metrics_excel(
    all_results,
    output_path: Path,
    fold_tag: str = '',
    split_name: str = 'TEST',
    class_names=None,
    n_boot: int = 5000,
):
    """将 DI Block 两两类别对比指标保存为单个 Excel 工作表。"""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df_metrics = _di_metrics_to_dataframe(all_results, fold_tag, split_name)
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_metrics.to_excel(writer, sheet_name='pairwise_metrics', index=False)
    return output_path, df_metrics


def summarize_di_block_metrics(
    diff_by_layer,
    labels,
    layer_ids,
    class_names,
    fold_tag: str = '',
    split_name: str = 'TEST',
    class_pairs=None,
    n_boot=5000,
    branch_configs=None,
    save_excel: bool = True,
    output_path: Path | None = None,
):
    """计算并打印 DI Block 前后特征指标（三分类：Effect Size / Fisher / Silhouette）。"""
    if class_pairs is None:
        class_pairs = DEFAULT_CLASS_PAIRS_3
    if branch_configs is None:
        branch_configs = DI_BRANCH_METRIC_CONFIGS

    labels = np.asarray(labels, dtype=int)
    header = f'\n=== {split_name} DI Block 特征指标'
    if fold_tag:
        header += f' | {fold_tag}'
    header += ' ==='
    print(header)
    print(f'样本数: {len(labels)} | 类别: {class_names} | bootstrap n={n_boot}')

    all_results = {}
    for cfg in branch_configs:
        rows = compute_di_block_metrics_table(
            diff_by_layer,
            labels,
            layer_ids,
            class_names,
            before_key=cfg['before_key'],
            after_key=cfg['after_key'],
            branch_name=cfg['branch'],
            class_pairs=class_pairs,
            n_boot=n_boot,
        )
        all_results[cfg['branch']] = rows

        print(f"\n--- {cfg['title']} ---")
        for row in rows:
            print(f"Layer {row['layer']} | {row['before_key']} -> {row['after_key']}")
            print('  两两类别对比 (Fisher / Silhouette / Effect Size):')
            print('    Before DI:')
            print('\n'.join(_format_pairwise_metric_line(rec) for rec in row['pairwise_before']))
            print('    After DI:')
            print('\n'.join(_format_pairwise_metric_line(rec) for rec in row['pairwise_after']))

    excel_path = None
    metrics_df = None
    if save_excel:
        n_classes = len(class_names)
        if output_path is None:
            fold_part = f'_{fold_tag}' if fold_tag else ''
            output_path = (
                DI_OUTPUT_DIR
                / f'di_block_metrics{fold_part}_{split_name.lower()}_classes{n_classes}.xlsx'
            )
        excel_path, metrics_df = save_di_block_metrics_excel(
            all_results,
            output_path,
            fold_tag=fold_tag,
            split_name=split_name,
            class_names=class_names,
            n_boot=n_boot,
        )
        print(f'\n指标已保存至 Excel（单工作表 pairwise_metrics）: {excel_path}')
        print(f'共 {len(metrics_df)} 行')

    return {
        'results': all_results,
        'excel_path': excel_path,
        'metrics_df': metrics_df,
    }

In [3]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

TSNE_RANDOM_STATE = 42
TSNE_PERPLEXITY = 30
TSNE_N_ITER = 1000
TSNE_LEARNING_RATE = 'auto'
PREPROCESS_STANDARDIZE = True
PREPROCESS_PCA = True
PCA_N_COMPONENTS = 50
PCA_VARIANCE_RATIO = None


def diff_key_to_features(diff_by_layer, layer_idx: int, key: str) -> np.ndarray:
    """将单键样本列表 (L, D) 转为 (N, D)，时间维做均值池化（等同 AdaptiveAvgPool1d）。"""
    arr = np.stack(diff_by_layer[layer_idx][key], axis=0)  # (N, L, D)
    return arr.mean(axis=1).astype(np.float64)


def _effective_perplexity(n_samples: int, perplexity: int) -> int:
    return int(min(perplexity, max(5, (n_samples - 1) // 3)))


def _resolve_pca_n_components(n_samples: int, n_features: int) -> int:
    max_comp = min(n_features, n_samples - 1)
    if max_comp < 1:
        return 1
    if PCA_N_COMPONENTS is not None:
        return int(min(PCA_N_COMPONENTS, max_comp))
    return int(min(10, max_comp))


def preprocess_features(features: np.ndarray) -> np.ndarray:
    X = np.asarray(features, dtype=np.float64)
    n_samples, n_features = X.shape
    if PREPROCESS_STANDARDIZE:
        X = StandardScaler().fit_transform(X)
    if PREPROCESS_PCA and X.shape[1] > 1:
        if PCA_VARIANCE_RATIO is not None:
            pca = PCA(n_components=PCA_VARIANCE_RATIO, random_state=TSNE_RANDOM_STATE)
            X = pca.fit_transform(X)
        else:
            n_comp = _resolve_pca_n_components(n_samples, X.shape[1])
            if n_comp < X.shape[1]:
                pca = PCA(n_components=n_comp, random_state=TSNE_RANDOM_STATE)
                X = pca.fit_transform(X)
    return X


def _preprocess_pipeline_tag() -> str:
    steps = []
    if PREPROCESS_STANDARDIZE:
        steps.append('StandardScaler')
    if PREPROCESS_PCA:
        steps.append('PCA')
    return ' → '.join(steps) + ' → ' if steps else ''


def run_tsne(features: np.ndarray, perplexity: int = TSNE_PERPLEXITY) -> np.ndarray:
    X = preprocess_features(features)
    n = X.shape[0]
    perp = _effective_perplexity(n, perplexity)
    tsne = TSNE(
        n_components=2,
        perplexity=perp,
        random_state=TSNE_RANDOM_STATE,
        max_iter=TSNE_N_ITER,
        learning_rate=TSNE_LEARNING_RATE,
        init='pca',
    )
    return tsne.fit_transform(X)


def _scatter_classes(ax, xy: np.ndarray, labels: np.ndarray, class_names, class_colors, show_legend: bool = False):
    labels = np.asarray(labels)
    for c in np.unique(labels):
        mask = labels == c
        ax.scatter(
            xy[mask, 0], xy[mask, 1],
            s=14, alpha=0.75,
            c=class_colors.get(int(c), '#333333'),
            label=class_names[int(c)] if int(c) < len(class_names) else str(c),
            edgecolors='none',
        )
    ax.grid(True, alpha=0.25)
    if show_legend:
        ax.legend(fontsize=8, frameon=False, loc='best')


def plot_di_branch_tsne_grid(
    diff_by_layer,
    labels,
    layer_ids,
    class_names,
    class_colors,
    fold_tag: str,
    branch: str,
    before_key: str,
    after_key: str,
    title: str,
    save_path: Path,
):
    """单折单分支：2 行（DI Block 前/后）× n_layers 列（各 encoder 层）。"""
    labels = np.asarray(labels)
    n_layers = len(layer_ids)
    stage_keys = [before_key, after_key]
    stage_names = ['Before DI', 'After DI']

    fig, axes = plt.subplots(2, n_layers, figsize=(2.8 * n_layers, 5.4))
    if n_layers == 1:
        axes = np.array(axes).reshape(2, 1)

    for col, layer_idx in enumerate(layer_ids):
        for row, (key, stage_name) in enumerate(zip(stage_keys, stage_names)):
            ax = axes[row, col]
            features = diff_key_to_features(diff_by_layer, layer_idx, key)
            xy = run_tsne(features)
            _scatter_classes(ax, xy, labels, class_names, class_colors, show_legend=(row == 0 and col == 0))
            if row == 0:
                ax.set_title(f'Layer {layer_idx}', fontsize=10)
            if col == 0:
                ax.set_ylabel(stage_name, fontsize=9)
            if row == 1:
                ax.set_xlabel('t-SNE 1', fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])

    handles, lbls = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, lbls, loc='upper center', ncol=len(class_names), bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(
        f'{title} | {_preprocess_pipeline_tag()}t-SNE — {fold_tag}',
        fontsize=12, y=1.06,
    )
    fig.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)


def _normalize_fold_indices(fold_indices):
    if fold_indices is None:
        return list(range(len(fold_dirs)))
    if isinstance(fold_indices, int):
        return [fold_indices]
    return list(fold_indices)


DEFAULT_LAYER_TSNE_PARAM_CONFIGS = (
    dict(param='mamba', before_key='mambaX', after_key='mambaX_', title='Mamba'),
    dict(param='gcn', before_key='gcnX', after_key='gcnX_', title='GCN'),
)


def plot_di_layer_params_tsne(
    diff_by_layer,
    labels,
    layer_idx: int,
    class_names,
    class_colors,
    fold_tag: str,
    save_path: Path,
    *,
    param_configs=None,
    stage_names=('Before DI', 'After DI'),
    subplot_size: float = 3.6,
    figsize: tuple[float, float] | None = None,
    subplot_title_fontsize: float = 10,
    stage_ylabel_fontsize: float = 9,
    axis_label_fontsize: float = 8,
    dpi: int = 300,
):
    """指定单层、两个参数：2 行（DI Block 前/后）× 2 列（两参数）t-SNE 对比图。"""
    if param_configs is None:
        param_configs = DEFAULT_LAYER_TSNE_PARAM_CONFIGS
    if len(param_configs) != 2:
        raise ValueError(f'param_configs 需要恰好 2 项，当前为 {len(param_configs)}')

    labels = np.asarray(labels)
    stage_keys = ('before_key', 'after_key')
    n_params = len(param_configs)
    if figsize is None:
        figsize = (subplot_size * n_params, subplot_size * 2)

    fig, axes = plt.subplots(2, n_params, figsize=figsize)
    if n_params == 1:
        axes = np.array(axes).reshape(2, 1)

    for col, cfg in enumerate(param_configs):
        before_key = cfg['before_key']
        after_key = cfg['after_key']
        param_title = cfg.get('title', cfg.get('param', f'Param {col}'))
        for row, (key_attr, stage_name) in enumerate(zip(stage_keys, stage_names)):
            ax = axes[row, col]
            key = cfg[key_attr]
            features = diff_key_to_features(diff_by_layer, layer_idx, key)
            xy = run_tsne(features)
            _scatter_classes(
                ax, xy, labels, class_names, class_colors,
                show_legend=False,
            )
            if row == 0:
                ax.set_title(param_title, fontsize=subplot_title_fontsize)
            if col == 0:
                ax.set_ylabel(stage_name, fontsize=stage_ylabel_fontsize)
            if row == 1:
                ax.set_xlabel('t-SNE 1', fontsize=axis_label_fontsize)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_box_aspect(1)

    fig.suptitle(
        f'Layer {layer_idx} | {_preprocess_pipeline_tag()}t-SNE — {fold_tag}',
        fontsize=12, y=1.02,
    )
    fig.tight_layout()
    save_path = Path(save_path)
    if save_path.suffix.lower() != '.svg':
        save_path = save_path.with_suffix('.svg')
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, format='svg', bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return save_path


def plot_di_layers_params_tsne(
    diff_by_layer,
    labels,
    layer_ids,
    class_names,
    class_colors,
    fold_tag: str,
    save_dir: Path,
    *,
    param_configs=None,
    n_classes_tag: str = 'classes3',
    filename_template: str = 'tsne_di_layer{layer_idx}_params_{fold_tag}_{n_classes_tag}.png',
    subplot_size: float = 3.6,
    dpi: int = 600,
    **plot_kwargs,
):
    """批量绘制多个指定层的两参数 DI Block 前后 t-SNE 对比图（每个子图为正方形）。"""
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    saved_paths = []
    for layer_idx in layer_ids:
        if layer_idx not in diff_by_layer:
            raise KeyError(f'diff_by_layer 中不存在 layer {layer_idx}')
        out_path = save_dir / filename_template.format(
            layer_idx=layer_idx, fold_tag=fold_tag, n_classes_tag=n_classes_tag,
        )
        saved_path = plot_di_layer_params_tsne(
            diff_by_layer,
            labels,
            layer_idx,
            class_names,
            class_colors,
            fold_tag=fold_tag,
            save_path=out_path,
            param_configs=param_configs,
            subplot_size=subplot_size,
            dpi=dpi,
            **plot_kwargs,
        )
        saved_paths.append(saved_path)
        print(f'  已保存: {saved_path}')
    return saved_paths


## 绘制三分类试验结果 散点图

### 基础参数配置

In [4]:
# ========== checkpoint 路径配置 ==========
PROJECT_NAME = (
    'CAEEG3_1s_00_EO_Aug-ADSSTNet-NCvsMCIvsAD-oneFreBanDAGnoRanCrop'
)
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / PROJECT_NAME

fold_dirs = sorted([d for d in CHECKPOINT_ROOT.iterdir() if d.is_dir()]) if CHECKPOINT_ROOT.is_dir() else []
print(f'CHECKPOINT_ROOT = {CHECKPOINT_ROOT}')
print(f'发现折目录 {len(fold_dirs)} 个: {[d.name for d in fold_dirs]}')

# ========== Differential Inhibition 分析配置 ==========
BASE_SEED = 38
ITRS = 5
FOLD_IDX = 0                   # 可选 0..4，对应 itr_000..itr_004
USE_SWA = True
CLASS_NAMES_3 = ['NC', 'MCI', 'AD']
CLASS_IDS_3 = [0, 1, 2]

# ========== t-SNE 可视化配置 ==========
font_path = PROJECT_ROOT / 'figs/fonts/TIMES.TTF'
if font_path.is_file():
    font_manager.fontManager.addfont(font_path)
PLOT_FONT = 'Times New Roman'
plt.rcParams.update({'font.family': PLOT_FONT, 'font.serif': [PLOT_FONT]})
CLASS_COLORS_3 = {0: '#1f77b4', 1: '#d62728', 2: '#ff7f0e'}  # NC / MCI / AD（MCI 与 AD 颜色对调）

COLLECT_CORRECT_ONLY = True    # True 时仅保留分类正确样本（测试集推荐）
BALANCE_CLASSES = True          # 按各类最少样本数均衡
COLLECT_FROM_TRAIN = False      # 是否同时从训练集收集
TRAIN_CORRECT_ONLY = False      # 训练集收集时是否仅保留正确样本
TRAIN_BALANCE_CLASSES = False   # 训练集收集时是否均衡类别

FOLD_TAG = f'itr_{FOLD_IDX:03d}'
fold_seed = BASE_SEED + FOLD_IDX + 1
fold_dir = CHECKPOINT_ROOT / FOLD_TAG
if not fold_dir.is_dir():
    raise FileNotFoundError(f'未找到折目录: {fold_dir}')
print(f'=== {FOLD_TAG} | seed={fold_seed} ===')
# 选择要绘制的折：单个 int、int 列表，或 None 表示全部折
TSNE_PLOT_FOLD_INDICES = [FOLD_IDX]   # 例如 [1]、[0, 2, 4]、None

DIFF_KEYS = ('mambaX', 'gcnX', 'mambaX_', 'gcnX_')

MODEL_ARGS_BASE = dict(
    task_name='supervised',
    model='ADSSTNet',
    data='CAEEG',
    root_path=str((PROJECT_ROOT / '/root/autodl-tmp/ADMamba/dataset/USTBMCI3_1.0s_0.0_EO_Aug').resolve()) + '/',
    augmentations='FreBandMix',
    selected_classes=[0, 1, 2],
    cross_val='mccv',
    use_augmentation=True,
    e_layers=0,
    d_model=20,
    expand=2,
    d_state=16,
    dt_rank=2,
    d_conv_in=4,
    d_conv=4,
    d_conv_out=4,
    bimamba=True,
    space=True,
    se_reduction=16,
    dim_UVA=4,
    topk_A0=4,
    embedding_dim=32,
    sample_adapt_A0=True,
    sample_adapt_Fre=True,
    projection_dim=128,
    dropout_head=0.7,
    num_heads=4,
    downsample_min_len=4,
    group_size=16,
    batch_size=256,
    num_workers=0,
    preserve_frontal_occipital=False,
)

A0_KWARGS = dict(
    use_abs=True,
    topk=MODEL_ARGS_BASE['topk_A0'],
    sym_mode='max',
    eps=1e-8,
    row_normalize=True,
    clip_negative=True,
    dataset_name=MODEL_ARGS_BASE['data'],
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DEVICE = {DEVICE}, USE_SWA = {USE_SWA}, FOLD_IDX = {FOLD_IDX}')

SEQ_EACH_LAYER = compute_seq_each_layer(INPUT_SEQ_LEN, DOWNSAMPLE_MIN_LEN)
print(f'INPUT_SEQ_LEN={INPUT_SEQ_LEN}, DOWNSAMPLE_MIN_LEN={DOWNSAMPLE_MIN_LEN}')
print(f'各层 seq_len: {SEQ_EACH_LAYER}')

CHECKPOINT_ROOT = /root/autodl-tmp/ADSST-Net/checkpoints/CAEEG3_1s_00_EO_Aug-ADSSTNet-NCvsMCIvsAD-oneFreBanDAGnoRanCrop
发现折目录 5 个: ['itr_000', 'itr_001', 'itr_002', 'itr_003', 'itr_004']
=== itr_000 | seed=39 ===
DEVICE = cuda, USE_SWA = True, FOLD_IDX = 0
INPUT_SEQ_LEN=128, DOWNSAMPLE_MIN_LEN=4
各层 seq_len: [128, 64, 32, 16, 8, 4]


### 初始化模型与加载 checkpoint，收集数据

In [5]:
# 1. 初始化模型（基于训练集计算 A0 与 seq_len 等参数）
model, fold_args = build_adsstnet_for_fold(fold_seed)

# 2. 加载训练好的 checkpoint
load_fold_checkpoint(model, fold_dir)

# 3. 加载训练集与测试集
train_data, train_loader = load_train_loader(fold_args)
test_data, test_loader = load_test_loader(fold_args)
print(f'训练集样本数: {len(train_data)}, 测试集样本数: {len(test_data)}')

# 4. 在测试集上收集 differential_visualization
diff_by_layer_test, label_list_test, layer_ids = collect_differential_visualization(
    model,
    test_loader,
    CLASS_NAMES_3,
    correct_only=COLLECT_CORRECT_ONLY,
    balance_classes=BALANCE_CLASSES,
    balance_seed=fold_seed,
    class_ids=CLASS_IDS_3,
    split_name='TEST',
)
summarize_differential_data(diff_by_layer_test, label_list_test, layer_ids, CLASS_NAMES_3, split_name='TEST')

train ids: [2, 5, 6, 7, 8, 11, 13, 14, 15, 17, 19, 20, 22, 27, 29, 31, 32, 33, 34, 35, 36, 38, 39, 41, 42, 44, 46, 48, 49, 50, 52, 54, 55, 56, 57, 59, 60, 61, 64, 67, 69, 70, 74, 75, 77, 78, 79, 80, 81, 82, 84, 88, 90, 91, 92, 93, 96, 97, 98, 99, 100, 101, 102, 104, 105, 106, 107, 108, 109, 111] (数量: 70)
train ids: [2, 5, 6, 7, 8, 11, 13, 14, 15, 17, 19, 20, 22, 27, 29, 31, 32, 33, 34, 35, 36, 38, 39, 41, 42, 44, 46, 48, 49, 50, 52, 54, 55, 56, 57, 59, 60, 61, 64, 67, 69, 70, 74, 75, 77, 78, 79, 80, 81, 82, 84, 88, 90, 91, 92, 93, 96, 97, 98, 99, 100, 101, 102, 104, 105, 106, 107, 108, 109, 111] (数量: 70)
test ids: [1, 3, 4, 9, 10, 12, 18, 23, 25, 26, 40, 43, 45, 58, 63, 71, 76, 83, 85, 87, 89, 103, 110] (数量: 23)
训练集样本数: 12424, 测试集样本数: 4089
collect_differential_visualization [TEST]: 保留正确分类样本 2012/4089
collect_differential_visualization [TEST]: 类别均衡采样 -> 每类 n=188 (NC: 1272->188, MCI: 552->188, AD: 188->188)

=== TEST 数据摘要 ===
总样本数: 564
  NC: 188
  MCI: 188
  AD: 188
Layer 0: n=564, shape

### 绘制 Differential Inhibition Block 前后的 t-SNE 特征分布图

- 通过 `TSNE_PLOT_FOLD_INDICES` 选择要绘制的折（如 `[1]`、`[0, 2, 4]` 或 `None` 表示全部）；若包含未收集的折，则仅对缺失折补充收集
- 每折输出 **2 张图**（Mamba 分支 / GCN 分支），每张图为 **2 × 6** 子图：
  - **行**：DI Block 前（上行）、DI Block 后（下行）
  - **列**：encoder 层 0–5
- 特征预处理与 t-SNE 参数参考 `tSEN_Visualization.ipynb`（StandardScaler → PCA → t-SNE）

In [6]:
# 每个分支一张图：2 行（DI Block 前/后）× 6 列（encoder 层）
BRANCH_PLOT_CONFIGS = [
    dict(branch='mamba', before_key='mambaX', after_key='mambaX_', title='Mamba branch'),
    dict(branch='gcn', before_key='gcnX', after_key='gcnX_', title='GCN branch'),
]

print(f'TSNE_OUTPUT_DIR = {DI_OUTPUT_DIR}')
print(f'TSNE_PLOT_FOLD_INDICES = {TSNE_PLOT_FOLD_INDICES}')

all_diff_data = {
    FOLD_TAG: {
        'diff_by_layer': diff_by_layer_test,
        'labels': np.asarray(label_list_test, dtype=int),
        'layer_ids': layer_ids,
        'fold_idx': FOLD_IDX,
    },
}
layer_ids_ref = layer_ids
print(f'复用已收集数据: {FOLD_TAG} | n={len(label_list_test)} | layers={layer_ids}')

# 若需绘制其它折，仅对缺失折补充收集
plot_fold_indices = _normalize_fold_indices(TSNE_PLOT_FOLD_INDICES)
missing_indices = [
    fold_idx for fold_idx in plot_fold_indices
    if fold_dirs[fold_idx].name not in all_diff_data
]

for fold_idx in missing_indices:
    fold_seed = BASE_SEED + fold_idx + 1
    fold_tag = fold_dirs[fold_idx].name
    fold_dir = fold_dirs[fold_idx]
    print(f'\n===== 补充收集 {fold_tag} | seed={fold_seed} =====')

    model, fold_args = build_adsstnet_for_fold(fold_seed)
    load_fold_checkpoint(model, fold_dir)
    _, test_loader = load_test_loader(fold_args)

    diff_by_layer, label_list, layer_ids_fold = collect_differential_visualization(
        model,
        test_loader,
        correct_only=COLLECT_CORRECT_ONLY,
        balance_classes=BALANCE_CLASSES,
        balance_seed=fold_seed,
        class_ids=CLASS_IDS_3,
        split_name='TEST',
    )
    if layer_ids_fold != layer_ids_ref:
        raise ValueError(f'{fold_tag}: encoder 层 {layer_ids_fold} 与已收集折不一致')

    all_diff_data[fold_tag] = {
        'diff_by_layer': diff_by_layer,
        'labels': np.asarray(label_list, dtype=int),
        'layer_ids': layer_ids_fold,
        'fold_idx': fold_idx,
    }
    print(f'  测试样本数: {len(label_list)}, encoder 层数: {len(layer_ids_fold)}')

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 2. 绘制选定折的 t-SNE 图
saved_paths = []

for fold_idx in plot_fold_indices:
    if fold_idx < 0 or fold_idx >= len(fold_dirs):
        raise IndexError(f'FOLD 索引 {fold_idx} 超出范围 [0, {len(fold_dirs) - 1}]')

    fold_tag = fold_dirs[fold_idx].name
    payload = all_diff_data[fold_tag]
    diff_by_layer = payload['diff_by_layer']
    labels = payload['labels']
    layer_ids = payload['layer_ids']

    print(f'\n===== 绘制 t-SNE: {fold_tag} | n={len(labels)} | layers={layer_ids} =====')

    for cfg in BRANCH_PLOT_CONFIGS:
        out_png = DI_OUTPUT_DIR / f"tsne_di_{cfg['branch']}_{fold_tag}_classes3.png"
        plot_di_branch_tsne_grid(
            diff_by_layer,
            labels,
            layer_ids,
            CLASS_NAMES_3,
            CLASS_COLORS_3,
            fold_tag=fold_tag,
            branch=cfg['branch'],
            before_key=cfg['before_key'],
            after_key=cfg['after_key'],
            title=cfg['title'],
            save_path=out_png,
        )
        saved_paths.append(out_png)
        print(f'  已保存: {out_png}')

print(f'\n共保存 {len(saved_paths)} 张 t-SNE 图 -> {DI_OUTPUT_DIR}')

TSNE_OUTPUT_DIR = /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction
TSNE_PLOT_FOLD_INDICES = [0]
复用已收集数据: itr_000 | n=564 | layers=[0, 1, 2, 3, 4, 5]

===== 绘制 t-SNE: itr_000 | n=564 | layers=[0, 1, 2, 3, 4, 5] =====
  已保存: /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction/tsne_di_mamba_itr_000_classes3.png
  已保存: /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction/tsne_di_gcn_itr_000_classes3.png

共保存 2 张 t-SNE 图 -> /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction


### 计算 Differential Inhibition Block 前后的特征指标

In [7]:
# 计算 DI Block 前后特征指标（三分类，两两类别对比：Fisher / Silhouette / Effect Size）
METRICS_N_BOOT = 5000

di_metrics_results_3class = summarize_di_block_metrics(
    diff_by_layer_test,
    label_list_test,
    layer_ids,
    CLASS_NAMES_3,
    fold_tag=FOLD_TAG,
    split_name='TEST',
    class_pairs=[(0, 1), (0, 2), (1, 2)],
    n_boot=METRICS_N_BOOT,
)
di_metrics_df_3class = di_metrics_results_3class['metrics_df']
print(f"Excel: {di_metrics_results_3class['excel_path']}")
di_metrics_df_3class.head(12)



=== TEST DI Block 特征指标 | itr_000 ===
样本数: 564 | 类别: ['NC', 'MCI', 'AD'] | bootstrap n=5000

--- Mamba branch ---
Layer 0 | mambaX -> mambaX_
  两两类别对比 (Fisher / Silhouette / Effect Size):
    Before DI:
      NC vs MCI   : Fisher=0.0260, Silhouette=0.0541, mean|g|=0.2676, boot=0.2820, 95%CI=[0.2346, 0.3298]
      NC vs AD    : Fisher=0.0906, Silhouette=0.1312, mean|g|=0.4751, boot=0.4802, 95%CI=[0.4380, 0.5203]
      MCI vs AD   : Fisher=0.0354, Silhouette=0.0332, mean|g|=0.3095, boot=0.3234, 95%CI=[0.2721, 0.3785]
    After DI:
      NC vs MCI   : Fisher=0.0179, Silhouette=0.0511, mean|g|=0.1959, boot=0.2178, 95%CI=[0.1815, 0.2567]
      NC vs AD    : Fisher=0.0664, Silhouette=0.1244, mean|g|=0.3385, boot=0.3573, 95%CI=[0.3171, 0.3980]
      MCI vs AD   : Fisher=0.0161, Silhouette=0.0298, mean|g|=0.1615, boot=0.1932, 95%CI=[0.1594, 0.2306]
Layer 1 | mambaX -> mambaX_
  两两类别对比 (Fisher / Silhouette / Effect Size):
    Before DI:
      NC vs MCI   : Fisher=0.0509, Silhouette=0.0925, mean

,split,fold,branch,layer,stage,feature_key,pair,c1,c2,fisher_score,silhouette_score,mean_abs_hedges_g,mean_abs_hedges_g_boot,ci_low,ci_high
0,TEST,itr_000,mamba,0,before,mambaX,NC vs MCI,0,1,0.026006,0.054060,0.267645,0.282022,0.234620,0.329845
1,TEST,itr_000,mamba,0,before,mambaX,NC vs AD,0,2,0.090564,0.131161,0.475116,0.480170,0.437950,0.520259
2,TEST,itr_000,mamba,0,before,mambaX,MCI vs AD,1,2,0.035369,0.033216,0.309477,0.323436,0.272130,0.378465
3,TEST,itr_000,mamba,0,after,mambaX_,NC vs MCI,0,1,0.017944,0.051133,0.195931,0.217816,0.181465,0.256659
4,TEST,itr_000,mamba,0,after,mambaX_,NC vs AD,0,2,0.066355,0.124436,0.338459,0.357278,0.317114,0.397955
5,TEST,itr_000,mamba,0,after,mambaX_,MCI vs AD,1,2,0.016110,0.029793,0.161536,0.193235,0.159443,0.230573
6,TEST,itr_000,mamba,1,before,mambaX,NC vs MCI,0,1,0.050900,0.092509,0.294482,0.310564,0.260934,0.360900
7,TEST,itr_000,mamba,1,before,mambaX,NC vs AD,0,2,0.170416,0.209543,0.593085,0.600914,0.549976,0.652870
8,TEST,itr_000,mamba,1,before,mambaX,MCI vs AD,1,2,0.092770,0.177195,0.464104,0.475077,0.428803,0.521626
9,TEST,itr_000,mamba,1,after,mambaX_,NC vs MCI,0,1,0.056321,0.097854,0.281305,0.309932,0.269516,0.353019


### 绘制单层的 特征对比 tSNE 图

In [8]:
# 绘制指定层的两个参数（Mamba / GCN）DI Block 前后 t-SNE 对比图
LAYER_TSNE_LAYER_IDS = [5]   # 指定 encoder 层，可改为单层如 [3] 或多层
LAYER_TSNE_PARAM_CONFIGS = [
    dict(param='mamba', before_key='mambaX', after_key='mambaX_', title='Mamba'),
    dict(param='gcn', before_key='gcnX', after_key='gcnX_', title='GCN'),
]

print(f'LAYER_TSNE_LAYER_IDS = {LAYER_TSNE_LAYER_IDS}')
print(f'输出目录: {DI_OUTPUT_DIR}')

layer_tsne_paths_3class = plot_di_layers_params_tsne(
    diff_by_layer_test,
    label_list_test,
    LAYER_TSNE_LAYER_IDS,
    CLASS_NAMES_3,
    CLASS_COLORS_3,
    fold_tag=FOLD_TAG,
    save_dir=DI_OUTPUT_DIR,
    param_configs=LAYER_TSNE_PARAM_CONFIGS,
    n_classes_tag='classes3',
    subplot_size=3.6
)
print(f'\n共保存 {len(layer_tsne_paths_3class)} 张单层参数对比 t-SNE 图')

LAYER_TSNE_LAYER_IDS = [5]
输出目录: /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction
  已保存: /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction/tsne_di_layer5_params_itr_000_classes3.svg

共保存 1 张单层参数对比 t-SNE 图


## 绘制二分类试验结果 散点图

### 基础参数配置

In [9]:
# ========== checkpoint 路径配置 ==========
PROJECT_NAME = (
    'CAEEG3_1s_00_EO_Aug-ADSSTNet-NCvsMCI-oneFreBanDAGnoRanCrop'
)
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / PROJECT_NAME

fold_dirs = sorted([d for d in CHECKPOINT_ROOT.iterdir() if d.is_dir()]) if CHECKPOINT_ROOT.is_dir() else []
print(f'CHECKPOINT_ROOT = {CHECKPOINT_ROOT}')
print(f'发现折目录 {len(fold_dirs)} 个: {[d.name for d in fold_dirs]}')

# ========== Differential Inhibition 分析配置 ==========
BASE_SEED = 38
ITRS = 5
FOLD_IDX = 0                   # 可选 0..4，对应 itr_000..itr_004
USE_SWA = True
CLASS_NAMES_2 = ['NC', 'MCI']
CLASS_IDS_2 = [0, 1]

# ========== t-SNE 可视化配置 ==========
font_path = PROJECT_ROOT / 'figs/fonts/TIMES.TTF'
if font_path.is_file():
    font_manager.fontManager.addfont(font_path)
PLOT_FONT = 'Times New Roman'
plt.rcParams.update({'font.family': PLOT_FONT, 'font.serif': [PLOT_FONT]})
CLASS_COLORS_2 = {0: '#1f77b4', 1: '#d62728'}

COLLECT_CORRECT_ONLY = True    # True 时仅保留分类正确样本（测试集推荐）
BALANCE_CLASSES = True          # 按各类最少样本数均衡
COLLECT_FROM_TRAIN = False      # 是否同时从训练集收集
TRAIN_CORRECT_ONLY = False      # 训练集收集时是否仅保留正确样本
TRAIN_BALANCE_CLASSES = False   # 训练集收集时是否均衡类别

FOLD_TAG = f'itr_{FOLD_IDX:03d}'
fold_seed = BASE_SEED + FOLD_IDX + 1
fold_dir = CHECKPOINT_ROOT / FOLD_TAG
if not fold_dir.is_dir():
    raise FileNotFoundError(f'未找到折目录: {fold_dir}')
print(f'=== {FOLD_TAG} | seed={fold_seed} ===')
# 选择要绘制的折：单个 int、int 列表，或 None 表示全部折
TSNE_PLOT_FOLD_INDICES = [FOLD_IDX]   # 例如 [1]、[0, 2, 4]、None

DIFF_KEYS = ('mambaX', 'gcnX', 'mambaX_', 'gcnX_')

MODEL_ARGS_BASE = dict(
    task_name='supervised',
    model='ADSSTNet',
    data='CAEEG',
    root_path=str((PROJECT_ROOT / '/root/autodl-tmp/ADMamba/dataset/USTBMCI3_1.0s_0.0_EO_Aug').resolve()) + '/',
    augmentations='FreBandMix',
    selected_classes=[0, 1],
    cross_val='mccv',
    use_augmentation=True,
    e_layers=0,
    d_model=20,
    expand=2,
    d_state=16,
    dt_rank=2,
    d_conv_in=4,
    d_conv=4,
    d_conv_out=4,
    bimamba=True,
    space=True,
    se_reduction=16,
    dim_UVA=4,
    topk_A0=4,
    embedding_dim=32,
    sample_adapt_A0=True,
    sample_adapt_Fre=True,
    projection_dim=128,
    dropout_head=0.7,
    num_heads=4,
    downsample_min_len=4,
    group_size=16,
    batch_size=256,
    num_workers=0,
    preserve_frontal_occipital=False,
)

A0_KWARGS = dict(
    use_abs=True,
    topk=MODEL_ARGS_BASE['topk_A0'],
    sym_mode='max',
    eps=1e-8,
    row_normalize=True,
    clip_negative=True,
    dataset_name=MODEL_ARGS_BASE['data'],
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DEVICE = {DEVICE}, USE_SWA = {USE_SWA}, FOLD_IDX = {FOLD_IDX}')

SEQ_EACH_LAYER = compute_seq_each_layer(INPUT_SEQ_LEN, DOWNSAMPLE_MIN_LEN)
print(f'INPUT_SEQ_LEN={INPUT_SEQ_LEN}, DOWNSAMPLE_MIN_LEN={DOWNSAMPLE_MIN_LEN}')
print(f'各层 seq_len: {SEQ_EACH_LAYER}')

CHECKPOINT_ROOT = /root/autodl-tmp/ADSST-Net/checkpoints/CAEEG3_1s_00_EO_Aug-ADSSTNet-NCvsMCI-oneFreBanDAGnoRanCrop
发现折目录 5 个: ['itr_000', 'itr_001', 'itr_002', 'itr_003', 'itr_004']
=== itr_000 | seed=39 ===
DEVICE = cuda, USE_SWA = True, FOLD_IDX = 0
INPUT_SEQ_LEN=128, DOWNSAMPLE_MIN_LEN=4
各层 seq_len: [128, 64, 32, 16, 8, 4]


### 初始化模型与加载 checkpoint，收集数据


In [10]:
# 1. 初始化模型（基于训练集计算 A0 与 seq_len 等参数）
model, fold_args = build_adsstnet_for_fold(fold_seed)

# 2. 加载训练好的 checkpoint
load_fold_checkpoint(model, fold_dir)

# 3. 加载训练集与测试集
train_data, train_loader = load_train_loader(fold_args)
test_data, test_loader = load_test_loader(fold_args)
print(f'训练集样本数: {len(train_data)}, 测试集样本数: {len(test_data)}')

# 4. 在测试集上收集 differential_visualization
diff_by_layer_test, label_list_test, layer_ids = collect_differential_visualization(
    model,
    test_loader,
    CLASS_NAMES_2,
    correct_only=COLLECT_CORRECT_ONLY,
    balance_classes=BALANCE_CLASSES,
    balance_seed=fold_seed,
    class_ids=CLASS_IDS_2,
    split_name='TEST',
)
summarize_differential_data(diff_by_layer_test, label_list_test, layer_ids, CLASS_NAMES_2, split_name='TEST')

train ids: [2, 5, 6, 8, 11, 13, 17, 19, 20, 22, 25, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 44, 46, 47, 48, 50, 51, 53, 54, 55, 57, 60, 61, 64, 65, 68, 69, 72, 73, 77, 78, 79, 81, 82, 86, 88, 92, 93, 94, 95, 96, 97, 98, 99, 100, 102, 104, 105, 106, 107, 109, 111] (数量: 63)
train ids: [2, 5, 6, 8, 11, 13, 17, 19, 20, 22, 25, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 44, 46, 47, 48, 50, 51, 53, 54, 55, 57, 60, 61, 64, 65, 68, 69, 72, 73, 77, 78, 79, 81, 82, 86, 88, 92, 93, 94, 95, 96, 97, 98, 99, 100, 102, 104, 105, 106, 107, 109, 111] (数量: 63)
test ids: [1, 3, 4, 9, 12, 18, 23, 26, 40, 43, 45, 63, 71, 76, 83, 85, 87, 89, 103, 110] (数量: 20)
训练集样本数: 11183, 测试集样本数: 3557
collect_differential_visualization [TEST]: 保留正确分类样本 2109/3557
collect_differential_visualization [TEST]: 类别均衡采样 -> 每类 n=683 (NC: 1426->683, MCI: 683->683)

=== TEST 数据摘要 ===
总样本数: 1366
  NC: 683
  MCI: 683
Layer 0: n=1366, shape=(128, 20) (L, D)
  mambaX   mean=-0.0151, std=0.2513, min=-20.7403, max=7.8895
  gcnX     m

### 绘制 Differential Inhibition Block 前后的 t-SNE 特征分布图

- 通过 `TSNE_PLOT_FOLD_INDICES` 选择要绘制的折（如 `[1]`、`[0, 2, 4]` 或 `None` 表示全部）；若包含未收集的折，则仅对缺失折补充收集
- 每折输出 **2 张图**（Mamba 分支 / GCN 分支），每张图为 **2 × 6** 子图：
  - **行**：DI Block 前（上行）、DI Block 后（下行）
  - **列**：encoder 层 0–5
- 特征预处理与 t-SNE 参数参考 `tSEN_Visualization.ipynb`（StandardScaler → PCA → t-SNE）

In [11]:
# 每个分支一张图：2 行（DI Block 前/后）× 6 列（encoder 层）
BRANCH_PLOT_CONFIGS = [
    dict(branch='mamba', before_key='mambaX', after_key='mambaX_', title='Mamba branch'),
    dict(branch='gcn', before_key='gcnX', after_key='gcnX_', title='GCN branch'),
]

print(f'TSNE_OUTPUT_DIR = {DI_OUTPUT_DIR}')
print(f'TSNE_PLOT_FOLD_INDICES = {TSNE_PLOT_FOLD_INDICES}')

all_diff_data = {
    FOLD_TAG: {
        'diff_by_layer': diff_by_layer_test,
        'labels': np.asarray(label_list_test, dtype=int),
        'layer_ids': layer_ids,
        'fold_idx': FOLD_IDX,
    },
}
layer_ids_ref = layer_ids
print(f'复用已收集数据: {FOLD_TAG} | n={len(label_list_test)} | layers={layer_ids}')

# 若需绘制其它折，仅对缺失折补充收集
plot_fold_indices = _normalize_fold_indices(TSNE_PLOT_FOLD_INDICES)
missing_indices = [
    fold_idx for fold_idx in plot_fold_indices
    if fold_dirs[fold_idx].name not in all_diff_data
]

for fold_idx in missing_indices:
    fold_seed = BASE_SEED + fold_idx + 1
    fold_tag = fold_dirs[fold_idx].name
    fold_dir = fold_dirs[fold_idx]
    print(f'\n===== 补充收集 {fold_tag} | seed={fold_seed} =====')

    model, fold_args = build_adsstnet_for_fold(fold_seed)
    load_fold_checkpoint(model, fold_dir)
    _, test_loader = load_test_loader(fold_args)

    diff_by_layer, label_list, layer_ids_fold = collect_differential_visualization(
        model,
        test_loader,
        correct_only=COLLECT_CORRECT_ONLY,
        balance_classes=BALANCE_CLASSES,
        balance_seed=fold_seed,
        class_ids=CLASS_IDS_2,
        split_name='TEST',
    )
    if layer_ids_fold != layer_ids_ref:
        raise ValueError(f'{fold_tag}: encoder 层 {layer_ids_fold} 与已收集折不一致')

    all_diff_data[fold_tag] = {
        'diff_by_layer': diff_by_layer,
        'labels': np.asarray(label_list, dtype=int),
        'layer_ids': layer_ids_fold,
        'fold_idx': fold_idx,
    }
    print(f'  测试样本数: {len(label_list)}, encoder 层数: {len(layer_ids_fold)}')

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 2. 绘制选定折的 t-SNE 图
saved_paths = []

for fold_idx in plot_fold_indices:
    if fold_idx < 0 or fold_idx >= len(fold_dirs):
        raise IndexError(f'FOLD 索引 {fold_idx} 超出范围 [0, {len(fold_dirs) - 1}]')

    fold_tag = fold_dirs[fold_idx].name
    payload = all_diff_data[fold_tag]
    diff_by_layer = payload['diff_by_layer']
    labels = payload['labels']
    layer_ids = payload['layer_ids']

    print(f'\n===== 绘制 t-SNE: {fold_tag} | n={len(labels)} | layers={layer_ids} =====')

    for cfg in BRANCH_PLOT_CONFIGS:
        out_png = DI_OUTPUT_DIR / f"tsne_di_{cfg['branch']}_{fold_tag}_classes2.png"
        plot_di_branch_tsne_grid(
            diff_by_layer,
            labels,
            layer_ids,
            CLASS_NAMES_2,
            CLASS_COLORS_2,
            fold_tag=fold_tag,
            branch=cfg['branch'],
            before_key=cfg['before_key'],
            after_key=cfg['after_key'],
            title=cfg['title'],
            save_path=out_png,
        )
        saved_paths.append(out_png)
        print(f'  已保存: {out_png}')

print(f'\n共保存 {len(saved_paths)} 张 t-SNE 图 -> {DI_OUTPUT_DIR}')

TSNE_OUTPUT_DIR = /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction
TSNE_PLOT_FOLD_INDICES = [0]
复用已收集数据: itr_000 | n=1366 | layers=[0, 1, 2, 3, 4, 5]

===== 绘制 t-SNE: itr_000 | n=1366 | layers=[0, 1, 2, 3, 4, 5] =====
  已保存: /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction/tsne_di_mamba_itr_000_classes2.png
  已保存: /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction/tsne_di_gcn_itr_000_classes2.png

共保存 2 张 t-SNE 图 -> /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction


### 计算 Differential Inhibition Block 前后的特征指标

In [12]:
# 计算 DI Block 前后特征指标（二分类，两两类别对比：Fisher / Silhouette / Effect Size）
METRICS_N_BOOT = 5000

di_metrics_results_2class = summarize_di_block_metrics(
    diff_by_layer_test,
    label_list_test,
    layer_ids,
    CLASS_NAMES_2,
    fold_tag=FOLD_TAG,
    split_name='TEST',
    class_pairs=[(0, 1)],
    n_boot=METRICS_N_BOOT,
)
di_metrics_df_2class = di_metrics_results_2class['metrics_df']
print(f"Excel: {di_metrics_results_2class['excel_path']}")
di_metrics_df_2class.head(12)



=== TEST DI Block 特征指标 | itr_000 ===
样本数: 1366 | 类别: ['NC', 'MCI'] | bootstrap n=5000

--- Mamba branch ---
Layer 0 | mambaX -> mambaX_
  两两类别对比 (Fisher / Silhouette / Effect Size):
    Before DI:
      NC vs MCI   : Fisher=0.0603, Silhouette=0.1185, mean|g|=0.3958, boot=0.4000, 95%CI=[0.3719, 0.4282]
    After DI:
      NC vs MCI   : Fisher=0.0584, Silhouette=0.1178, mean|g|=0.3778, boot=0.3809, 95%CI=[0.3523, 0.4091]
Layer 1 | mambaX -> mambaX_
  两两类别对比 (Fisher / Silhouette / Effect Size):
    Before DI:
      NC vs MCI   : Fisher=0.0964, Silhouette=0.1203, mean|g|=0.4140, boot=0.4175, 95%CI=[0.3908, 0.4447]
    After DI:
      NC vs MCI   : Fisher=0.0826, Silhouette=0.1138, mean|g|=0.3615, boot=0.3657, 95%CI=[0.3426, 0.3898]
Layer 2 | mambaX -> mambaX_
  两两类别对比 (Fisher / Silhouette / Effect Size):
    Before DI:
      NC vs MCI   : Fisher=0.0351, Silhouette=0.0391, mean|g|=0.2488, boot=0.2565, 95%CI=[0.2337, 0.2785]
    After DI:
      NC vs MCI   : Fisher=0.0870, Silhouette=0.1004

,split,fold,branch,layer,stage,feature_key,pair,c1,c2,fisher_score,silhouette_score,mean_abs_hedges_g,mean_abs_hedges_g_boot,ci_low,ci_high
0,TEST,itr_000,mamba,0,before,mambaX,NC vs MCI,0,1,0.060263,0.118541,0.395848,0.400015,0.371861,0.428231
1,TEST,itr_000,mamba,0,after,mambaX_,NC vs MCI,0,1,0.058410,0.117845,0.377774,0.380929,0.352314,0.409147
2,TEST,itr_000,mamba,1,before,mambaX,NC vs MCI,0,1,0.096380,0.120310,0.413996,0.417489,0.390826,0.444685
3,TEST,itr_000,mamba,1,after,mambaX_,NC vs MCI,0,1,0.082563,0.113785,0.361458,0.365719,0.342619,0.389784
4,TEST,itr_000,mamba,2,before,mambaX,NC vs MCI,0,1,0.035076,0.039088,0.248810,0.256486,0.233731,0.278480
5,TEST,itr_000,mamba,2,after,mambaX_,NC vs MCI,0,1,0.086989,0.100419,0.391390,0.398094,0.373344,0.422431
6,TEST,itr_000,mamba,3,before,mambaX,NC vs MCI,0,1,0.052778,0.075867,0.295365,0.302577,0.279204,0.326410
7,TEST,itr_000,mamba,3,after,mambaX_,NC vs MCI,0,1,0.197413,0.156320,0.672417,0.675992,0.638765,0.712772
8,TEST,itr_000,mamba,4,before,mambaX,NC vs MCI,0,1,0.090776,0.212927,0.387308,0.390335,0.363795,0.417503
9,TEST,itr_000,mamba,4,after,mambaX_,NC vs MCI,0,1,0.336391,0.267457,0.859701,0.861732,0.820194,0.904657


### 绘制单层的 特征对比 tSNE 图

In [13]:
# 绘制指定层的两个参数（Mamba / GCN）DI Block 前后 t-SNE 对比图（二分类）
LAYER_TSNE_LAYER_IDS = [5]   # 指定 encoder 层，可改为单层如 [3] 或多层
LAYER_TSNE_PARAM_CONFIGS = [
    dict(param='mamba', before_key='mambaX', after_key='mambaX_', title='Mamba'),
    dict(param='gcn', before_key='gcnX', after_key='gcnX_', title='GCN'),
]

print(f'LAYER_TSNE_LAYER_IDS = {LAYER_TSNE_LAYER_IDS}')
print(f'输出目录: {DI_OUTPUT_DIR}')

layer_tsne_paths_2class = plot_di_layers_params_tsne(
    diff_by_layer_test,
    label_list_test,
    LAYER_TSNE_LAYER_IDS,
    CLASS_NAMES_2,
    CLASS_COLORS_2,
    fold_tag=FOLD_TAG,
    save_dir=DI_OUTPUT_DIR,
    param_configs=LAYER_TSNE_PARAM_CONFIGS,
    n_classes_tag='classes2',
    subplot_size=3.6
)
print(f'\n共保存 {len(layer_tsne_paths_2class)} 张单层参数对比 t-SNE 图')


LAYER_TSNE_LAYER_IDS = [5]
输出目录: /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction
  已保存: /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction/tsne_di_layer5_params_itr_000_classes2.svg

共保存 1 张单层参数对比 t-SNE 图


## 散点图 图例绘制

In [14]:
# 单独绘制三分类 t-SNE 散点图图例（NC / MCI / AD）
def plot_class_legend_figure(
    class_names,
    class_colors: dict[int, str],
    save_path: Path,
    *,
    class_ids: list[int] | None = None,
    ncol: int | None = None,
    fontsize: float = 12,
    marker_size: float = 10,
    scatter_alpha: float = 0.75,
    dpi: int = 600,
    figsize: tuple[float, float] | None = None,
    frameon: bool = False,
):
    """单独绘制类别图例图（与 t-SNE 散点图配色、圆点样式一致）。"""
    if class_ids is None:
        class_ids = sorted(class_colors.keys())
    ncol = ncol or len(class_ids)
    if figsize is None:
        figsize = (max(2.5, 0.95 * ncol), 0.5)

    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    handles = [
        plt.Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor=class_colors.get(int(c), '#333333'),
            markeredgecolor='none',
            alpha=scatter_alpha,
            markersize=marker_size,
            linestyle='None',
            label=class_names[int(c)] if int(c) < len(class_names) else str(c),
        )
        for c in class_ids
    ]
    ax.legend(
        handles=handles,
        loc='center',
        ncol=ncol,
        frameon=frameon,
        fontsize=fontsize,
        handlelength=1.2,
        handletextpad=0.6,
        columnspacing=1.2,
    )
    fig.tight_layout()
    save_path = Path(save_path)
    if save_path.suffix.lower() != '.svg':
        save_path = save_path.with_suffix('.svg')
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, format='svg', bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return save_path

    
legend_path_3class = plot_class_legend_figure(
    CLASS_NAMES_3,
    CLASS_COLORS_3,
    DI_OUTPUT_DIR / 'legend_tsne_classes3.svg',
    class_ids=[0, 1, 2],
    ncol=3,
    fontsize=12,
    marker_size=10,
    dpi=600,
)
print(f'已保存: {legend_path_3class}')
legend_path_3class


已保存: /root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction/legend_tsne_classes3.svg


PosixPath('/root/autodl-tmp/ADSST-Net/figs/Weight Visualization Analysis/Residual-Guided Differential Interaction/legend_tsne_classes3.svg')